# Qwen3.8-27B BF16 on Kaggle TPU v5e-8 — Text + Vision Production Showcase

## English

This notebook is the public **production-style showcase and reproducibility entry point** for running the original `Qwen/Qwen3.8-27B` BF16 checkpoint on a Kaggle **TPU v5e-8** runtime with MaxText/JAX. It demonstrates the qualified text path on all 8 TPU devices and surfaces frozen, reviewer-verifiable vision qualification evidence produced by the same Qwen3.8 MaxText port.

The implementation keeps the original BF16 model weights. It does **not** use GGUF, weight quantization, CPU offload, or a modified model architecture. The engineering work is in the MaxText compatibility/mapping/runtime path required to load and execute Qwen3.8-27B correctly on the target TPU environment.

This is a **production-style demo**, not a permanently hosted service, SLA-backed deployment, HA service, or multi-tenant platform. Kaggle sessions are ephemeral and cold XLA compilation is intentionally reported separately from warm steady-state latency.

## Tiếng Việt

Notebook này là **production-style showcase và entry point tái lập public** để chạy checkpoint BF16 nguyên bản `Qwen/Qwen3.8-27B` trên Kaggle **TPU v5e-8** bằng MaxText/JAX. Notebook trình diễn text path đã qualify trên đủ 8 TPU devices và hiển thị frozen vision qualification evidence có thể kiểm tra độc lập từ cùng Qwen3.8 MaxText port.

Triển khai giữ nguyên model weight BF16 gốc. Notebook **không** dùng GGUF, không quantization, không CPU offload và không thay đổi kiến trúc model. Phần engineering nằm ở compatibility/mapping/runtime path của MaxText để Qwen3.8-27B có thể load và thực thi đúng trên TPU mục tiêu.

Đây là **production-style demo**, không phải dịch vụ host thường trực, không có SLA, HA hay multi-tenant. Kaggle session là ephemeral và cold XLA compilation được tách riêng khỏi warm steady-state latency.

## What this notebook proves / Notebook này chứng minh điều gì

### English

The showcase is organized as a sequence of fail-closed gates. A reviewer can independently inspect the model attachment, offline support bundle, pinned MaxText source identity, TPU topology, original BF16 parameter load, semantic generation, deterministic rerun, warm text latency, and the frozen multimodal acceptance record.

| Capability / gate | Qualification target |
|---|---|
| Model format | Original BF16 safetensors |
| Accelerator | Kaggle TPU v5e-8, exactly 8 devices |
| Runtime | JAX 0.11.0 + pinned MaxText source |
| Sharding | 8-way FSDP path |
| Text semantics | deterministic semantic sanity cases |
| Text reproducibility | exact token IDs + exact decoded text |
| Warm text latency | prefill + decode ms/token |
| Vision primary sample | `Code?` + `731` card → `731` |
| Vision second sample | `Code?` + `482` card → `482` |
| Vision reproducibility | exact token sequence on rerun |
| Vision warm benchmark | 10/10 accepted repetitions |

### Tiếng Việt

Showcase được tổ chức thành chuỗi fail-closed gates. Reviewer có thể kiểm tra độc lập model attachment, offline support bundle, MaxText source identity đã pin, topology TPU, quá trình load BF16 parameter nguyên bản, semantic generation, deterministic rerun, warm text latency và frozen multimodal acceptance record.

Các số liệu cold compile không được trộn với warm latency. Notebook cũng không tuyên bố throughput hay speedup nào chưa được đo trực tiếp.

## Before you run / Trước khi chạy

### English

1. Select **TPU v5e-8** as the Kaggle accelerator.
2. Keep **Internet = OFF**. This notebook is intentionally offline-first.
3. Attach Kaggle Model `dangkhoa2016/qwen-qwen3-8-27b` → `Transformers / default / 1`.
4. Attach Dataset `dangkhoa2016/qwen3-8-27b-tpu-v5e8-offline-support`.
5. Confirm the expected mounts:
   - model: `/kaggle/input/models/dangkhoa2016/qwen-qwen3-8-27b/transformers/default/1`
   - support: `/kaggle/input/qwen3-8-27b-tpu-v5e8-offline-support`
6. Use **Restart Session → Run All** for a fresh reproduction. The first compile can be very long; warm timings are reported separately.

### Tiếng Việt

1. Chọn **TPU v5e-8** làm accelerator trên Kaggle.
2. Giữ **Internet = OFF**. Notebook được thiết kế offline-first.
3. Attach Kaggle Model `dangkhoa2016/qwen-qwen3-8-27b` → `Transformers / default / 1`.
4. Attach Dataset `dangkhoa2016/qwen3-8-27b-tpu-v5e8-offline-support`.
5. Kiểm tra đúng mount path của model và support dataset như phía trên.
6. Dùng **Restart Session → Run All** để tái lập fresh run. Cold compile đầu tiên có thể rất lâu; warm latency được báo riêng.

> **Publication note / Ghi chú publication:** visible outputs below are preserved from the accepted fresh TPU text run only when the code-cell source is byte-identical. Vision results are explicitly labeled **Frozen qualification evidence** and are loaded from immutable JSON artifacts in the attached support dataset; they are not presented as a new live run.

## Runtime architecture / Kiến trúc runtime

| Layer / Lớp | Qualified configuration / Cấu hình đã qualify |
|---|---|
| Model | Qwen3.8-27B, original BF16 safetensors |
| Host environment | Kaggle Python 3.12 |
| TPU | v5e-8, 8 JAX devices |
| Core runtime | JAX `0.11.0`, JAXLIB `0.11.0` |
| MaxText source | `f0b2abe3f6f3736c5579673f42b5cbc17b327fd3` |
| Text patch SHA256 | `92fd84c9a08f1f017b9e84987847ad18734fc3b6453f989f604180adc4f96951` |
| Text+vision patch SHA256 | `418edb5b5e3ab0707649d9af10c0f6b7471b7a2705101268cc8a59de2c9bc4e2` |
| Parameter restore | dynamic safetensors → target TPU sharding |
| Scan layers | disabled in the accepted baseline |
| Network dependency | none when support dataset is attached |

**English:** The notebook avoids materializing a full random 27B parameter tree before checkpoint restore. The accepted dynamic safetensors path maps and places tensors directly into the target sharding layout.

**Tiếng Việt:** Notebook tránh materialize toàn bộ random parameter tree 27B trước khi restore checkpoint. Dynamic safetensors path đã qualify map và đặt tensor trực tiếp vào target sharding layout.

## Step 1 — Fail-closed model and TPU environment preflight / Kiểm tra model và môi trường TPU

### English

Validate the exact attached model path and configure TPU bootstrap environment variables before importing JAX/XLA-sensitive stacks.

### Tiếng Việt

Xác minh đúng model path đã attach và cấu hình TPU bootstrap environment trước khi import các stack có thể khởi tạo JAX/XLA.

### Expected evidence / Kết quả cần thấy

- `MODEL_DIR=.../qwen-qwen3-8-27b/transformers/default/1`
- `TPU bootstrap env configured`

In [ ]:
# 1. Fail-closed preflight: attached model + TPU environment variables
# IMPORTANT: set TPU env before importing JAX / transformers stacks that can initialize XLA.
import os
from pathlib import Path

MODEL_DIR = Path('/kaggle/input/models/dangkhoa2016/qwen-qwen3-8-27b/transformers/default/1')
assert MODEL_DIR.is_dir(), f'Model not attached at: {MODEL_DIR}'
assert (MODEL_DIR / 'config.json').is_file(), 'Missing config.json'
assert (MODEL_DIR / 'model.safetensors.index.json').is_file(), 'Missing safetensors index'

os.environ.update({
    'PJRT_DEVICE': 'TPU',
    'TPU_SKIP_MDS_QUERY': '1',
    'TPU_HOST_BOUNDS': '1,1,1',
    'TPU_CHIPS_PER_HOST_BOUNDS': '2,4,1',
    'TPU_WORKER_HOSTNAMES': os.uname().nodename,
    'TPU_WORKER_ID': '0',
    'TPU_ACCELERATOR_TYPE': 'v5e-8',
    'JAX_PLATFORMS': 'tpu,cpu',
})
print('MODEL_DIR=', MODEL_DIR)
print('TPU bootstrap env configured')

## Step 2 — Offline dependency integrity gate / Cổng kiểm tra dependency offline

### English

Verify pinned bootstrap wheels by SHA256 and preserve Kaggle's accepted JAX/JAXLIB stack instead of replacing the live TPU runtime.

### Tiếng Việt

Xác minh pinned bootstrap wheels bằng SHA256 và giữ nguyên JAX/JAXLIB đã qualify của Kaggle thay vì thay thế TPU runtime đang hoạt động.

### Expected evidence / Kết quả cần thấy

- `Core versions: jax=0.11.0 / jaxlib=0.11.0`
- `DEPENDENCY_GATE=PASS`

In [ ]:
# 2. Offline-first dependency gate — never replace Kaggle's JAX/TPU stack.
import hashlib
import importlib.metadata as md
import subprocess, sys
from pathlib import Path

SUPPORT_DIR = Path('/kaggle/input/datasets/dangkhoa2016/qwen3-8-27b-tpu-v5e8-offline-support')
WHEEL_INFO = {
    'pathwaysutils': ('0.1.12', 'pathwaysutils-0.1.12-py3-none-any.whl', '3aab3cd936ea9076c469b731cbdca43a453736594c65bdcee4b274c10ce51c34'),
    'aqtp': ('0.9.0', 'aqtp-0.9.0-py3-none-any.whl', '5efdccd24657e149d3ac17599bb3d4eb88297fdc6aab14eaba0c70625057a257'),
    'drjax': ('0.2.1', 'drjax-0.2.1-py3-none-any.whl', '9ec93541a7ba0af864d599cd3e14770a19e291a6291b8b79ddf6b92e8b173494'),
    'tokamax': ('0.0.14', 'tokamax-0.0.14-py3-none-any.whl', 'a10a91bbb6ddf406bb7006cb1a27af59a6faaa6cb06fe788249dd10d9eaac409'),
    'xprof': ('2.23.1', 'xprof-2.23.1-py3-none-manylinux_2_27_x86_64.whl', '96701f868b7a736a789d0a4b5efeeb269feaed63348117707e853a72e006c722'),
    'einshape': ('1.0', 'einshape-1.0-py3-none-any.whl', '42da4c2dea3a27f87ee45a7cee5072a636b97cb184bb07bf5d6412ba0ff7b965'),
    'typeguard': ('2.13.3', 'typeguard-2.13.3-py3-none-any.whl', '5e3e3be01e887e7eafae5af63d1f36c849aaa94e3a0112097312aabfa16284f1'),
    'tiktoken': ('0.14.0', 'tiktoken-0.14.0-cp312-cp312-manylinux_2_28_x86_64.whl', '7896eea257fe497a2b7134474d909156c6744ce8da35bce88011a960e008aa0d'),
}

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

for dist, (ver, wheel_name, expected_sha) in WHEEL_INFO.items():
    try: current=md.version(dist)
    except md.PackageNotFoundError: current=None
    if current != ver:
        wheel = SUPPORT_DIR / 'wheels' / wheel_name
        if not wheel.is_file():
            wheel = SUPPORT_DIR / wheel_name
        if wheel.is_file():
            assert sha256(wheel) == expected_sha, f'SHA mismatch: {wheel}'
            print(f'Installing offline {wheel_name} (current={current})')
            subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps',str(wheel)])
        else:
            print(f'Offline wheel missing; installing pinned {dist}=={ver} from package index')
            subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps',f'{dist}=={ver}'])
    assert md.version(dist) == ver, (dist, md.version(dist), ver)

CORE_EXPECTED = {'jax': '0.11.0', 'jaxlib': '0.11.0', 'transformers': '5.12.1'}
versions = {k: md.version(k) for k in CORE_EXPECTED}
print('Core versions:', versions)
mismatches={k:(versions[k],v) for k,v in CORE_EXPECTED.items() if versions[k] != v}
assert not mismatches, f'Fail-closed: core runtime differs from acceptance environment: {mismatches}. Do not pip-replace Kaggle JAX/JAXLIB in a live TPU runtime.'
print('DEPENDENCY_GATE=PASS', 'offline_support='+str(SUPPORT_DIR.is_dir()))

## Step 3 — Pinned MaxText source and patch provenance / MaxText source và patch provenance đã pin

### English

Restore the exact offline MaxText archive, verify archive and real patch-file digests, and require the Qwen3.8 model configuration marker. No base64 patch payload is embedded in notebook code.

### Tiếng Việt

Restore đúng offline MaxText archive, kiểm tra digest của archive và patch file thật, đồng thời bắt buộc Qwen3.8 model configuration marker tồn tại. Code notebook không nhúng patch bằng base64.

### Expected evidence / Kết quả cần thấy

- `MaxText commit: f0b2abe...`
- `Patch SHA256: 92fd84c9...`
- `Qwen3.8 patch: READY`

In [ ]:
# 3. Offline MaxText bootstrap with exact source/patch provenance.
import hashlib, json, shutil, tarfile
from pathlib import Path

MAXTEXT_COMMIT = 'f0b2abe3f6f3736c5579673f42b5cbc17b327fd3'
MAXTEXT_DIR = Path('/kaggle/working/maxtext-qwen38-src')
PATCH_SHA256 = '2ec8241c93b3a3e1d54e97e85eace644d27f90ba2147c1bc8059bf67998c38cf'
OFFLINE_ARCHIVE_SHA256 = '45aef84519fbd21698a6528bf6a58b4a7f66d8605c60371cf2a0e45fc3b613ee'

def file_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

assert SUPPORT_DIR.is_dir(), f'Offline support dataset not attached: {SUPPORT_DIR}'
PATCH_PATH = SUPPORT_DIR / 'maxtext-qwen38-production-text-vision.patch'
assert PATCH_PATH.is_file(), f'Missing MaxText patch provenance file: {PATCH_PATH}'
assert file_sha256(PATCH_PATH) == PATCH_SHA256, 'MaxText patch SHA mismatch'

archive_name = 'maxtext-f0b2abe-qwen38-text-vision-patched.tar.gz'
archive_candidates = [
    SUPPORT_DIR / 'src' / archive_name,
    SUPPORT_DIR / archive_name,
    SUPPORT_DIR / (archive_name + '.blob'),
]
archive = next((p for p in archive_candidates if p.is_file()), archive_candidates[0])
assert archive.is_file(), f'Missing patched MaxText archive: {archive_candidates}'
assert file_sha256(archive) == OFFLINE_ARCHIVE_SHA256, 'Offline MaxText archive SHA mismatch'

if MAXTEXT_DIR.exists(): shutil.rmtree(MAXTEXT_DIR)
MAXTEXT_DIR.mkdir(parents=True)
with tarfile.open(archive,'r:gz') as tf: tf.extractall(MAXTEXT_DIR)
meta=json.loads((MAXTEXT_DIR/'QWEN38_OFFLINE_BUILD.json').read_text())
assert meta['maxtext_commit'] == MAXTEXT_COMMIT
assert meta['patch_sha256'] == PATCH_SHA256
assert meta.get('scope') == 'Qwen3.8-27B text+vision combined production candidate'
assert 'Q/K RMSNorm' in meta.get('qk_norm_fix','')
source_mode='offline-support-dataset'

marker = MAXTEXT_DIR / 'src/maxtext/configs/models/qwen3.8-27b.yml'
assert marker.exists(), 'Qwen3.8 MaxText patch marker missing'
print('MaxText commit:', MAXTEXT_COMMIT)
print('Patch SHA256:', PATCH_SHA256)
print('Source mode:', source_mode)
print('Qwen3.8 patch: READY')

In [ ]:
# 3b. Clone this project's reproducible inference examples from GitHub.
# The notebook never substitutes local Kaggle files when git clone fails.
import subprocess, hashlib, json, shutil
from pathlib import Path
EXAMPLES_REPO_URL = 'https://github.com/dangkhoa2016/Qwen3-8-27B-TPU-v5e8-Production-Showcase.git'
EXAMPLES_COMMIT = '7fb60c6bae3112b8927b86917bc29749ea9f13ab'
EXAMPLES_REPO_DIR = Path('/kaggle/working/qwen38-production-example-repo')
if EXAMPLES_REPO_DIR.exists():
    shutil.rmtree(EXAMPLES_REPO_DIR)
subprocess.run(['git', 'clone', '--no-checkout', EXAMPLES_REPO_URL, str(EXAMPLES_REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(EXAMPLES_REPO_DIR), 'checkout', '--detach', EXAMPLES_COMMIT], check=True)
actual = subprocess.check_output(['git','-C',str(EXAMPLES_REPO_DIR),'rev-parse','HEAD'], text=True).strip()
assert actual == EXAMPLES_COMMIT, (actual, EXAMPLES_COMMIT)
EXAMPLES_DIR = EXAMPLES_REPO_DIR / 'examples'
assets = json.loads((EXAMPLES_DIR/'assets_manifest.json').read_text())['assets']
for item in assets:
    asset = EXAMPLES_REPO_DIR / item['path']
    assert asset.is_file(), asset
    digest = hashlib.sha256(asset.read_bytes()).hexdigest()
    assert digest == item['sha256'], ('asset mismatch', asset.name)
print('EXAMPLES_REPO_COMMIT=', actual)
print('EXAMPLES_ASSET_SHA256_GATE=PASS; count=',len(assets))


## Step 4 — Checkpoint structure audit before TPU-heavy work / Audit checkpoint trước tác vụ TPU nặng

### English

Inspect the attached safetensors index and expected source checkpoint structure before constructing the expensive TPU engine.

### Tiếng Việt

Kiểm tra safetensors index và cấu trúc source checkpoint đã attach trước khi dựng TPU engine tốn tài nguyên.

### Expected evidence / Kết quả cần thấy

- `checkpoint audit`
- `fail closed before TPU-heavy initialization`

In [ ]:
# 4. Source checkpoint audit before TPU-heavy work.
import json
from pathlib import Path

cfg = json.loads((MODEL_DIR / 'config.json').read_text())
index = json.loads((MODEL_DIR / 'model.safetensors.index.json').read_text())
shards = sorted(MODEL_DIR.glob('model-*.safetensors'))
print('architecture:', cfg.get('architectures'))
print('model_type:', cfg.get('model_type'))
print('safetensors shards:', len(shards))
print('declared total bytes:', index.get('metadata', {}).get('total_size'))
assert len(shards) == 18, f'Expected 18 shards, found {len(shards)}'
assert index.get('metadata', {}).get('total_size') == 55_562_855_904
print('CHECKPOINT_AUDIT=PASS')

## Step 5 — Initialize JAX and prove the 8-device topology / Khởi tạo JAX và chứng minh topology 8 thiết bị

### English

Initialize the TPU backend only after environment and dependency gates pass, then require exactly eight TPU devices.

### Tiếng Việt

Chỉ khởi tạo TPU backend sau khi environment/dependency gates đã pass và bắt buộc đúng tám TPU devices.

### Expected evidence / Kết quả cần thấy

- `TPU device count=8`
- `MaxText import/config success`

In [ ]:
# 5. Initialize JAX TPU and patched MaxText.
import sys, time
os.environ['MAXTEXT_REPO_ROOT'] = str(MAXTEXT_DIR)
sys.path.insert(0, str(MAXTEXT_DIR / 'src'))

import jax
import jax.numpy as jnp
from transformers import AutoTokenizer
from maxtext.configs import pyconfig
from maxtext.inference.maxengine.maxengine import MaxEngine
from maxtext.utils.globals import MAXTEXT_REPO_ROOT

print('backend=', jax.default_backend())
print('device_count=', jax.device_count())
print('local_device_count=', jax.local_device_count())
assert jax.default_backend() == 'tpu'
assert jax.device_count() == 8
print('TPU_GATE=PASS')

## Step 6 — Build one multimodal engine and load BF16 once / Dựng một multimodal engine và load BF16 một lần

### English
Use one Qwen3.8-27B multimodal MaxText engine for **both text and vision**. The engine uses `PREFILL=256` and `TARGET=384`, loads the original BF16 safetensors exactly once, and keeps one parameter tree resident for the rest of the notebook.

### Tiếng Việt
Dùng một multimodal MaxText engine Qwen3.8-27B cho **cả text lẫn vision**. Engine dùng `PREFILL=256`, `TARGET=384`, chỉ load original BF16 safetensors đúng một lần và giữ duy nhất một parameter tree resident cho phần còn lại của notebook.

Expected: `MODEL_LOAD=PASS`, `ONE_ENGINE_POLICY=PASS`.

In [ ]:
# 6. One multimodal engine, one BF16 load.
from maxtext.multimodal import processor as mm_processor
from maxtext.multimodal import processor_qwen3_omni as qwen_mm

PREFILL_LEN = 256
MAX_TARGET_LEN = 384
VISION_SEED_IMAGE = EXAMPLES_DIR / 'images/vision_test_card_731_256x192.png'
assert VISION_SEED_IMAGE.is_file(), VISION_SEED_IMAGE
args = [
    '', f'{MAXTEXT_REPO_ROOT}/src/maxtext/configs/base.yml',
    'model_name=qwen3.8-27b',
    f'tokenizer_path={MODEL_DIR}',
    f'load_parameters_path={MODEL_DIR}',
    'source_checkpoint_layout=safetensors_dynamic',
    'override_model_config=true',
    'scan_layers=false',
    'use_multimodal=true',
    f'image_path={VISION_SEED_IMAGE}',
    'prompt=Code?',
    f'max_prefill_predict_length={PREFILL_LEN}',
    f'max_target_length={MAX_TARGET_LEN}',
    'per_device_batch_size=0.125',
    'attention=dot_product',
    'dtype=bfloat16', 'weight_dtype=bfloat16',
    'ici_data_parallelism=1', 'ici_fsdp_parallelism=-1', 'ici_tensor_parallelism=1',
    'ici_autoregressive_parallelism=1',
    'async_checkpointing=false', 'enable_checkpointing=false',
    'base_output_directory=/tmp/maxtext_output',
    'run_name=qwen38-kaggle-text-vision-showcase',
]
t0=time.time(); config=pyconfig.initialize(args); engine=MaxEngine(config); construct_s=time.time()-t0
print(f'engine_construct_seconds={construct_s:.3f}')
t0=time.time(); params=engine.load_params(); load_s=time.time()-t0
print(f'load_params_seconds={load_s:.3f}')
assert getattr(config,'use_multimodal',False) is True
print('MODEL_LOAD=PASS')
print('ONE_ENGINE_POLICY=PASS')

## Step 7 — Shared text + vision generation helpers / Helper generation dùng chung cho text + vision

### English
Both paths use the same engine and BF16 parameter tree. Text requests pass `images=None`; vision requests use the checkpoint-compatible Qwen2-VL-style preprocessing, image-token fusion, MRoPE positions, and the official chat template with thinking disabled for deterministic reviewer-facing answers.

### Tiếng Việt
Cả hai path dùng chung engine và BF16 parameter tree. Text request truyền `images=None`; vision request dùng preprocessing tương thích checkpoint, image-token fusion, MRoPE positions và official chat template với thinking disabled để tạo câu trả lời deterministic dễ review.

In [ ]:
# 7. Shared deterministic helpers.
import numpy as np, statistics
from IPython.display import display, Image as IPImage

tok = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True, trust_remote_code=False)
pad_id = tok.pad_token_id if tok.pad_token_id is not None else (tok.eos_token_id or 0)

def _decode(prefix, first, true_length, max_decode_steps):
    state=engine.init_decode_state(); state=engine.insert(prefix,state,slot=0)
    out=[int(jax.device_get(first.data)[0,0])]; step_s=[]
    for _ in range(max_decode_steps-1):
        t=time.time(); state,res=engine.generate(params=params,decode_state=state,algorithm='greedy'); jax.block_until_ready(res.data); step_s.append(time.time()-t)
        tid=int(jax.device_get(res.data)[0,0]); out.append(tid)
        if tid == tok.eos_token_id: break
    return out, tok.decode(out, skip_special_tokens=False), step_s

def generate_text(prompt: str, max_decode_steps: int=24):
    rendered=tok.apply_chat_template([{'role':'user','content':prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids=tok(rendered, add_special_tokens=False)['input_ids']
    assert len(ids) <= PREFILL_LEN, (len(ids), PREFILL_LEN, prompt)
    padded=np.asarray(ids+[pad_id]*(PREFILL_LEN-len(ids)),dtype=np.int32)
    t=time.time(); prefix,first=engine.prefill(params=params,padded_tokens=jnp.asarray(padded),true_length=len(ids),algorithm='greedy'); jax.block_until_ready(first.data); pre=time.time()-t
    out,text,steps=_decode(prefix,first,len(ids),max_decode_steps)
    return {'prompt':prompt,'prompt_token_count':len(ids),'generated_token_ids':out,'generated_text':text,'prefill_seconds':pre,'decode_step_seconds':steps}

def generate_vision(image_path, prompt: str, max_decode_steps: int=160):
    image_path=Path(image_path); assert image_path.is_file(), image_path
    pd=config._pydantic_config.model_copy(update={'image_path':str(image_path),'prompt':prompt})
    cfg=pyconfig.HyperParameters(pd)
    po=mm_processor.preprocess_mm_data(cfg)
    off=int(mm_processor.get_image_offsets(config=cfg,processor_output=po))
    content=f'<|vision_start|><|image_pad|><|vision_end|>{prompt}'
    rendered=tok.apply_chat_template([{'role':'user','content':content}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    base=tok(rendered,add_special_tokens=False)['input_ids']
    text_budget=PREFILL_LEN-off
    assert len(base) <= text_budget, {'text_tokens':len(base),'image_offset':off,'prefill':PREFILL_LEN}
    padded=np.asarray(base+[pad_id]*(text_budget-len(base)),dtype=np.int32)
    fused=mm_processor.prepare_text_for_image_fusion(tokens=padded,config=cfg,processor_output=po)
    true_length=len(base)+off
    mask=(np.arange(len(fused))[None,:] < true_length).astype(np.int32)
    pos,delta=qwen_mm.get_rope_index(
        input_ids=fused[None,:], image_grid_thw=po.pixel_grid_thw, attention_mask=mask,
        spatial_merge_size=cfg.spatial_merge_size_for_vit,
        position_id_per_seconds=cfg.position_id_per_seconds, config=cfg,
    )
    t=time.time(); prefix,first=engine.prefill(
        params=params,padded_tokens=jnp.asarray(fused,dtype=jnp.int32),positions=jnp.asarray(pos),mrope_deltas=jnp.asarray(delta),
        images=jnp.asarray(po.pixel_values),true_length=true_length,algorithm='greedy'); jax.block_until_ready(first.data); pre=time.time()-t
    max_decode_steps=min(max_decode_steps, MAX_TARGET_LEN-true_length)
    out,text,steps=_decode(prefix,first,true_length,max_decode_steps)
    return {'image':str(image_path),'prompt':prompt,'grid':po.pixel_grid_thw.tolist(),'image_offset_tokens':off,'fused_true_length':true_length,'generated_token_ids':out,'generated_text':text,'prefill_seconds':pre,'decode_step_seconds':steps}

print('SHARED_GENERATION_HELPERS=READY')

## Step 8 — Live text semantic gates / Live text semantic gates

Run three compact text sanity checks on the **same multimodal engine**. These checks verify that enabling multimodal support does not break ordinary text generation.

In [ ]:
PROMPTS=['2+2','France capital','Ruby language']
text_rows=[]
for p in PROMPTS:
    r=generate_text(p)
    low=r['generated_text'].lower()
    ok=('4' in r['generated_text']) if p=='2+2' else ('paris' in low if p=='France capital' else ('ruby' in low and ('programming' in low or 'language' in low)))
    r['semantic_check']=bool(ok); text_rows.append(r)
    print(json.dumps({k:r[k] for k in ('prompt','prompt_token_count','generated_text','semantic_check','prefill_seconds')},ensure_ascii=False,indent=2))
assert all(r['semantic_check'] for r in text_rows)
print('TEXT_SEMANTIC_GATE=PASS')

## Step 9 — Text determinism + warm latency / Text determinism + warm latency

Repeat the accepted text outputs exactly, then benchmark only the warmed compiled path. Cold XLA compilation is kept separate from steady-state latency.

In [ ]:
text_repro=[]
for ref in text_rows:
    cur=generate_text(ref['prompt'])
    text_repro.append({'prompt':ref['prompt'],'token_ids_match':cur['generated_token_ids']==ref['generated_token_ids'],'text_match':cur['generated_text']==ref['generated_text']})
assert all(x['token_ids_match'] and x['text_match'] for x in text_repro)
print('TEXT_DETERMINISM=PASS',json.dumps(text_repro))

prefill=[]; decode=[]
for _ in range(5):
    r=generate_text('France capital',max_decode_steps=8); prefill.append(r['prefill_seconds']); decode.extend(r['decode_step_seconds'])
def pct(xs,p):
    ys=sorted(xs); return ys[min(len(ys)-1,max(0,int(round((len(ys)-1)*p))))]
text_latency={'repetitions':5,'prefill_ms':{'mean':statistics.mean(prefill)*1000,'median':statistics.median(prefill)*1000,'p95':pct(prefill,.95)*1000},'decode_ms_per_token':{'mean':statistics.mean(decode)*1000,'median':statistics.median(decode)*1000,'p95':pct(decode,.95)*1000}}
print('TEXT_WARM_LATENCY='+json.dumps(text_latency,indent=2))

### Vision test image fidelity

The six regression scene assets are native **256×192** and are displayed at native size below. Enlarging their notebook display to 512px only interpolates pixels; it does not improve the source. The original regression images are deliberately unchanged so the previously verified inference checks remain comparable. See the native **1024×768** image gallery at the end for clear, high-resolution visual inspection; those three additional photos are *not* claimed as accepted model inference.

## Step 10 — Live strict vision acceptance: 731 + 482 / Live strict vision acceptance: 731 + 482

### English
Run two independent synthetic visual probes **live in this notebook**. These are strict engineering gates for image-conditioned execution, not a broad VLM benchmark. The `731` probe is rerun exactly to prove deterministic token behavior.

### Tiếng Việt
Chạy trực tiếp hai visual probe độc lập. Đây là strict engineering gates cho image-conditioned execution, không phải benchmark VLM diện rộng. Probe `731` được chạy lại chính xác để chứng minh deterministic token behavior.

In [ ]:
vision_cards=[]
for name,expected in [('vision_test_card_731_256x192.png','731'),('vision_test_card_482_256x192.png','482')]:
    path=EXAMPLES_DIR/'images'/name
    display(IPImage(filename=str(path),width=256,height=192))
    r=generate_vision(path,'Read the large number in this image. Answer only the digits.',max_decode_steps=12)
    r['expected']=expected; r['semantic_check']=expected in r['generated_text']; vision_cards.append(r)
    print(json.dumps({k:r[k] for k in ('image','grid','fused_true_length','generated_text','semantic_check','prefill_seconds')},ensure_ascii=False,indent=2))
assert all(x['semantic_check'] for x in vision_cards)
rerun_731=generate_vision(EXAMPLES_DIR/'images/vision_test_card_731_256x192.png','Read the large number in this image. Answer only the digits.',max_decode_steps=12)
assert rerun_731['generated_token_ids']==vision_cards[0]['generated_token_ids']
assert rerun_731['generated_text']==vision_cards[0]['generated_text']
print('VISION_731=PASS')
print('VISION_482=PASS')
print('VISION_DETERMINISM=PASS')

## Step 11 — Live six-scene capability gallery / Live capability gallery 6 cảnh

### English
Run six richer images spanning 2D infographic/diagram, 3D scenes, and real-world-style scenes. The bundled images are **256×192 q40 JPEG derivatives** of the selected originals. Their automatic OCR hit counts are therefore reported as diagnostics, not publication gates. Scene/object grounding and the complete model response remain visible to the reviewer.

### Tiếng Việt
Chạy sáu ảnh phong phú hơn gồm infographic/diagram 2D, cảnh 3D và cảnh theo phong cách đời thực. Các ảnh bundle hiện là **JPEG q40 256×192** từ originals đã chọn. Vì vậy hit-count OCR tự động chỉ là diagnostic, không phải publication gate. Reviewer vẫn thấy đầy đủ scene/object grounding và toàn bộ câu trả lời của model.

In [ ]:
rich_spec=json.loads((EXAMPLES_DIR/'vision_showcase_cases.json').read_text())['cases']
assert len(rich_spec)==6
rich_results=[]
for case in rich_spec:
    path=EXAMPLES_DIR/'images'/case['image']
    print('\n===',case['id'],'===')
    display(IPImage(filename=str(path),width=256))
    r=generate_vision(path,case['prompt'],max_decode_steps=160)
    low=r['generated_text'].lower(); hits={x:(x.lower() in low) for x in case.get('checks',[])}
    r.update({'id':case['id'],'kind':case.get('kind'),'checks':hits,'check_hit_count':sum(hits.values()),'check_count':len(hits)})
    rich_results.append(r)
    print(r['generated_text'])
    print('OCR_DIAGNOSTIC',r['check_hit_count'],'/',r['check_count'],hits)
assert len(rich_results)==6
# Fail-closed scene-grounding sanity checks that do not depend only on tiny OCR text.
by_id={x['id']:x for x in rich_results}
assert all(k in by_id['vision-2d-dashboard']['generated_text'] for k in ('128','94%','6.4','42'))
assert 'robot' in by_id['vision-3d-robotics']['generated_text'].lower()
assert 'dock b2' in by_id['vision-3d-warehouse']['generated_text'].lower()
print('RICH_VISION_SCENE_GROUNDING=PASS')
print('RICH_VISION_OCR_DIAGNOSTIC_TOTAL=',sum(x['check_hit_count'] for x in rich_results),'/',sum(x['check_count'] for x in rich_results))

## Step 12 — Machine-readable current-run evidence / Machine-readable evidence của current run

Persist current-run text, strict vision, rich-scene diagnostics, provenance, and runtime timing under `/kaggle/working`.

In [ ]:
artifact={
  'status':'PASS','backend':jax.default_backend(),'device_count':jax.device_count(),
  'maxtext_commit':MAXTEXT_COMMIT,'text_vision_patch_sha256':PATCH_SHA256, 'examples_repo_url':EXAMPLES_REPO_URL, 'examples_commit':EXAMPLES_COMMIT,'maxtext_source_mode':source_mode,
  'core_versions':versions,'model_dir':str(MODEL_DIR),'one_engine_one_bf16_load':True,
  'engine_construct_seconds':construct_s,'load_params_seconds':load_s,
  'text_semantic_results':text_rows,'text_reproducibility':text_repro,'text_warm_latency':text_latency,
  'vision_strict_cards':vision_cards,'vision_731_rerun':rerun_731,'rich_vision_results':rich_results,
  'rich_ocr_diagnostic':{'hits':sum(x['check_hit_count'] for x in rich_results),'checks':sum(x['check_count'] for x in rich_results)},
}
out=Path('/kaggle/working/qwen38_27b_tpu_v5e8_text_vision_fresh_20261018.json')
out.write_text(json.dumps(artifact,indent=2,ensure_ascii=False)+'\n')
print('WROTE',out)
print('CURRENT_RUN_EVIDENCE=PASS')

## Results summary / Tổng kết kết quả

| Gate | Expected result |
|---|---:|
| Original BF16 safetensors | PASS |
| TPU v5e-8 topology | 8 devices |
| Combined MaxText patch/archive SHA gate | PASS |
| Qwen3 hybrid Q/K RMSNorm semantics | fixed in combined artifact |
| One engine / one BF16 load | PASS |
| Text semantic + determinism | PASS |
| Live vision `731` / `482` | PASS |
| Live vision determinism | PASS |
| Six rich vision scenes | executed live |
| Rich OCR hit rate | diagnostic only |

### English
The notebook demonstrates one original-BF16 Qwen3.8-27B runtime serving text and vision on Kaggle TPU v5e-8. The strict 731/482 probes gate image conditioning; the six richer scenes demonstrate capability breadth. Tiny-text OCR misses in the rich gallery are reported rather than hidden because the bundled gallery images are heavily reduced q40 256×192 derivatives.

### Tiếng Việt
Notebook chứng minh một runtime Qwen3.8-27B original-BF16 duy nhất phục vụ cả text và vision trên Kaggle TPU v5e-8. Hai probe 731/482 là strict gate cho image conditioning; sáu cảnh phong phú dùng để trình diễn độ rộng năng lực. Các lỗi OCR chữ nhỏ được báo thẳng vì gallery bundle hiện là derivative q40 256×192 đã giảm mạnh độ phân giải.

## Limitations and reproduction notes / Giới hạn và ghi chú tái lập

### English
- Cold XLA compilation can take many minutes per new static shape; warm latency is reported separately.
- The rich gallery assets are reduced derivatives; do not interpret their tiny-text OCR hit rate as a model-wide accuracy benchmark.
- The Q/K RMSNorm correction is included in the SHA-verified combined text+vision MaxText artifact; no runtime hot-patch is required in a fresh notebook.
- Internet remains OFF; the notebook uses the attached model and offline support dataset only.

### Tiếng Việt
- Cold XLA compilation có thể mất nhiều phút cho mỗi static shape mới; warm latency được báo riêng.
- Rich gallery dùng ảnh derivative đã giảm mạnh; không dùng tiny-text OCR hit rate của bộ này làm benchmark accuracy toàn model.
- Bản sửa Q/K RMSNorm nằm trong combined text+vision MaxText artifact có SHA verification; fresh notebook không cần runtime hot-patch.
- Internet giữ OFF; notebook chỉ dùng attached model và offline support dataset.

### High-resolution image gallery (source examples from Git)

The following are original native **1024×768** images stored in the cloned project repo. These are visual inspection assets, *not* new inference results. The earlier six benchmark scenes are natively 256×192.

In [ ]:
from PIL import Image as PILImage
for name in ['northstar_office_supply_invoice.jpg','warm_desk_still_life_with_business_card.jpg','cozy_daylight_workspace_with_city_views.jpg']:
    path=EXAMPLES_DIR/'images'/name
    with PILImage.open(path) as im:
        assert im.size == (1024, 768), (name, im.size)
    print('Native 1024×768:',name)
    display(IPImage(filename=str(path), width=1024, height=768))
print('HIGH_RES_SOURCE_GALLERY=READY (no inference acceptance asserted)')
